### Библиотеки

In [1]:
import grpc
import json
import pandas as pd

from backend.grpc.contracts import contracts_pb2, contracts_pb2_grpc

### Тестовые данные

In [2]:
train_data = pd.read_csv("../../data/train.csv")
test_data = pd.read_csv("../../data/test.csv")

with open("../../data/titanic_config.json", "r") as f:
    run_config = json.load(f)

user_id = "17"

### Проверка сервисов

In [3]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:
    health_stub = contracts_pb2_grpc.HealthServiceStub(channel)

    resp = await health_stub.CheckApp(contracts_pb2.HealthRequest())
    print("Health:", resp.app)

    resp = await health_stub.CheckS3(contracts_pb2.S3HealthRequest(bucket="models-bucket"))
    print("S3:", resp.s3)

Health: ok
S3: ok


### Доступные модели

In [4]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)

    resp = await model_stub.ListAvailableModels(
        contracts_pb2.Empty(
        )
    )
    print("Available models:\n\n", *resp.models)

Available models:

 name: "ElasticNet"
hyperparameters {
  alpha: 1
  l1_ratio: 0.5
}
 name: "GradientBoostingRegressor"
hyperparameters {
  learning_rate: 0.1
  max_depth: 3
  n_estimators: 100
}



### Загрузка данных

In [5]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    storage_stub = contracts_pb2_grpc.DatasetRegistryServiceStub(channel)  
    csv_bytes = train_data.to_csv(index=False).encode("utf-8")

    upload_resp = await storage_stub.UploadFile(
        contracts_pb2.UploadRequest(
            user_id=user_id,
            filename="train.csv",
            file_bytes=csv_bytes,
        )
    )
    print("Upload result:", upload_resp)
    data_id = upload_resp.data_id

Upload result: user_id: "17"
data_id: "c134e957-1466-46ce-993b-3eb297c711d5"
filename: "train.csv"



### Датасеты пользователя

In [6]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    storage_stub = contracts_pb2_grpc.DatasetRegistryServiceStub(channel)  
    user_stub = contracts_pb2_grpc.UserStorageServiceStub(channel)
    
    print("\nStorage usage:")
    usage = await storage_stub.GetUsage(contracts_pb2.UserRequest(user_id=user_id))
    print(f"Usage: {round(usage.usage_mb, 2)} MB")

    print("\nList user datasets:")
    datasets = await user_stub.ListDatasets(contracts_pb2.UserRequest(user_id=user_id))
    for ds in datasets.datasets:
        print(f"- {ds.data_id} ({ds.name})")


Storage usage:
Usage: 0.04 MB

List user datasets:
- c134e957-1466-46ce-993b-3eb297c711d5 (train.csv)


### Обучение модели

In [7]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    print("\nTrain model")

    train_resp = await model_stub.Train(
        contracts_pb2.TrainRequest(
            user_id=user_id,
            data_id=data_id,
            run_config_json=json.dumps(run_config),
        )
    )
    print("Model trained:", train_resp.model_name)
    print("Metrics:", train_resp.metrics_json)


Train model
Model trained: GradientBoostingRegressor___learning_rate___0_1___max_depth___3___n_estimators___150_
Metrics: [{"name": "R2", "value": 0.714492841351754}, {"name": "MSE", "value": 0.0026830515094594244}, {"name": "MAE", "value": 0.021021215833341}]


### Инференс обученной модели

In [8]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    print("\nPredict")
    input_data = test_data.drop(columns=['Fare'], axis=1).to_dict('records')
    predict_resp = await model_stub.Predict(
        contracts_pb2.PredictRequest(
            user_id=user_id,
            data_id=data_id,
            model_name=train_resp.model_name,
            input_data_json=json.dumps(input_data),
        )
    )
    print("Predictions:", predict_resp.predictions)


Predict
Predictions: [5.712058649484075, 12.131082774424778, 9.38649593099083, 8.940270814933365, 17.653868098770825, 11.02005827693993, 7.384633034862738, 28.327722824728415, 16.35644814854318, 21.11702231912855, 9.617716078956416, 42.1305712532175, 77.44300464542542, 24.379620262176022, 59.336908972916135, 24.595829763996164, 65.59635687424836, 8.857882738804001, 15.256716508062862, 5.818271519117181, 49.763893036651574, 15.835297929805119, 68.47533176321863, 115.76471972444799, 111.45524992846532, 22.90776636686409, 114.4560034792784, 8.857882738804001, 42.75867086465705, 10.614300637904872, 30.997564467618858, 45.51021059882278, 26.034461765954774, 27.903744895222403, 70.39524824884322, 18.04505362907168, 9.540145952526066, 8.24961468187544, 9.063865289351753, 9.617716078956416, 11.108299135452057, 43.33196325542843, 6.144291846727358, 13.605451717041772, 67.49313212032246, 9.063865289351753, 65.68533424741975, 7.462203161293089, 91.71649162442858, 21.238985678369293, 52.358563425

### Удаление датасета

In [9]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    storage_stub = contracts_pb2_grpc.DatasetRegistryServiceStub(channel)  

    del_resp = await storage_stub.DeleteFile(
        contracts_pb2.FileRequest(
            user_id=user_id,
            data_id=data_id,
        )
    )
    print("Delete status:", del_resp.status)

Delete status: deleted


### Удаление модели

In [10]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    print("\n=== Delete model ===")
    del_resp = await model_stub.Delete(
        contracts_pb2.DeleteRequest(
            user_id=user_id,
            data_id=data_id,
            model_name=train_resp.model_name,
        )
    )
    print("Delete status:", del_resp.status)


=== Delete model ===
Delete status: deleted


___